# Dong Boundary Condition: Steady-State Convergence study computing Kovasznay Flow

First test problem for validating the outflow B.C. by S. Dong. The exact solution for the velocity and pressure fields of the Kovasznay flow is given by:

>$$  \begin{align*} 
u &= 1 - \textrm{e}^{\lambda x} \cos{(2 \pi y)}, \\
v &= \frac{\lambda}{2 \pi} \textrm{e}^{\lambda x} \sin{(2 \pi y)}, \\
p &= \frac{1}{2} (1 - \textrm{e}^{2 \lambda x})
\end{align*}$$

where $\lambda = \frac{1}{2 \nu} - \sqrt{\frac{1}{4 \nu^2} + 4 \pi^2}$ with $\nu = \frac{1}{40}$.

In [1]:
//#r "../../src/L4-application/BoSSSpad/bin/Release/net5.0/BoSSSpad.dll"
//#r "../../src/L4-application/BoSSSpad/bin/Debug/net5.0/BoSSSpad.dll"
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [2]:
using BoSSS.Solution.NSECommon;

In [3]:
var myBatch = ExecutionQueues[1];
myBatch

RuntimeLocation,win\amd64
AdditionalEnvironmentVars,[ ]
DeploymentBaseDirectory,\\dc3\userspace\smuda\hpccluster\binaries
DeployRuntime,True
Name,FDY-WindowsHPC
DotnetRuntime,dotnet
Username,FDY\smuda
NumOfAdditionalServiceCores,0
NumOfAdditionalServiceCoresMPISerial,0
NumOfServiceCoresPerMPIprocess,0
ServerName,DC3


In [4]:
BoSSSshell.WorkflowMgm.Init("KovasznayFlow_ConvStudy", myBatch);

## Grid generation

In [5]:
bool UseDongBC = false;

In [6]:
int[] Resolutions = new int[] { 1, 2, 4, 8, 16, 32, 64, 128, 256 };
IGridInfo[] Grids = new IGridInfo[Resolutions.Length];

for(int i = 0; i < Resolutions.Length; i++) {
    int Res = Resolutions[i];
    string GridName = $"KovasznayFlow_{Res}x{2*Res}_DongBC";

    // IGridInfo cachedGrid = wmg.Grids.FirstOrDefault(grid => grid.Name == GridName);
    IGridInfo cachedGrid = null;
    if(cachedGrid == null) {
        
        // must create new Grid
        double xend = -0.1;
        double[] xNodes = GenericBlas.Linspace(-0.5, xend, Res + 1);
        double[] yNodes = GenericBlas.Linspace(-0.5, 0.5, (2 * Res) + 1);
        
        var grd = Grid2D.Cartesian2DGrid(xNodes, yNodes, periodicY:true);
        grd.Name = GridName;
        
        grd.DefineEdgeTags(delegate(Vector X) {
            string ret = null;
            if((X.x + 0.5).Abs() <= 1e-8)
                ret = IncompressibleBcType.Velocity_Inlet.ToString();
            if((X.x - xend).Abs() <= 1e-8) {
                ret = UseDongBC ? IncompressibleBcType.Dong_OutFlow.ToString() : IncompressibleBcType.Velocity_Inlet.ToString();
            }
            return ret;
        });        
        
        Grids[i] = wmg.SaveGrid(grd);
        
    } else {
        //Console.WriteLine($"type: {cachedGrid.GetType()}, is IGridInfo? {cachedGrid is IGridInfo}");
        Console.WriteLine("Grid already found in database - identifid by name " + GridName);
        Grids[i] = cachedGrid;
    }
    
}

## Initial Values

In [7]:
Formula KovasznayFlow_u = new Formula(
    "VelX",
    false,
    "double VelX(double[] X) { " + 
    "double lambda = 20.0 - 2.0 * Math.Sqrt(Math.PI.Pow2() + 100.0); "  + 
    "double velX = 1.0 - (Math.Exp(lambda * X[0]) * Math.Cos(2.0 * Math.PI * X[1]));" +
    "return velX; } "
);

In [8]:
Formula KovasznayFlow_v = new Formula(
    "VelY",
    false,
    "double VelY(double[] X) { " + 
    "double lambda = 20.0 - 2.0 * Math.Sqrt(Math.PI.Pow2() + 100.0); "  + 
    "double velY = (lambda/(2.0 * Math.PI)) * (Math.Exp(lambda * X[0]) * Math.Sin(2.0 * Math.PI * X[1]));" +
    "return velY; } "
);

In [9]:
Formula KovasznayFlow_p = new Formula(
    "Pres",
    false,
    "double Pres(double[] X) { " + 
    "double lambda = 20.0 - 2.0 * Math.Sqrt(Math.PI.Pow2() + 100.0); "  + 
    "double Pres = 0.5 * (1.0 - Math.Exp(2.0 * lambda * X[0]));" +
    "return Pres; } "
);

## Setting up control 

In [10]:
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Application.XNSE_Solver.DongBoundaryConditionTests;

In [11]:
int[] polyDeg = { 2, 3 };

In [19]:
List<XNSE_Test_Control> Controls = new List<XNSE_Test_Control>();
Controls.Clear();

In [18]:
bool SolveSystem = true;

In [20]:
foreach (int k in polyDeg) {    // loop over poylnomial degrees
foreach (var grd in Grids) {    // loop over grids
//for (int res = 0; res < Resolutions.Length; res++) {
    var C = new XNSE_Test_Control();

    C.SetDGdegree(k);

    // physical parameters
    double rhoSpc = 1.0;
    C.PhysicalParameters.rho_A = rhoSpc;
    double muSpc = 1.0 / 40.0;
    C.PhysicalParameters.mu_A = muSpc;

    C.PhysicalParameters.IncludeConvection = true;

    C.SetGrid(grd);
    int J = grd.NumberOfCells * ((((k + 1)*(k + 1)) + (k + 1)) / 2);
    

    // initial conditions
    if (!SolveSystem) {
        C.AddInitialValue("VelocityX#A", KovasznayFlow_u);
        C.AddInitialValue("VelocityY#A", KovasznayFlow_v);
        C.AddInitialValue("Pressure#A", KovasznayFlow_p);
    }

    // boundary conditions
    C.AddBoundaryValue(IncompressibleBcType.Velocity_Inlet.ToString(), "VelocityX#A", KovasznayFlow_u);
    C.AddBoundaryValue(IncompressibleBcType.Velocity_Inlet.ToString(), "VelocityY#A", KovasznayFlow_v);

    if (UseDongBC) {
        C.UseManufacturedComps = true;
        //C.AddBoundaryValue(IncompressibleBcType.Dong_OutFlow.ToString(), "VelocityX#A", KovasznayFlow_u);
        //C.AddBoundaryValue(IncompressibleBcType.Dong_OutFlow.ToString(), "VelocityY#A", KovasznayFlow_v);
    } 

    C.SkipSolveAndEvaluateResidual = !SolveSystem;
    //C.CutCellQuadratureType = XQuadFactoryHelper.MomentFittingVariants.OneStepGaussAndStokes;

    //C.NonLinearSolver.SolverCode = NonLinearSolverCode.Picard;
    //C.NonLinearSolver.ConvergenceCriterion = 1e-9;
    C.NonLinearSolver.Globalization = Newton.GlobalizationOption.LineSearch;
    C.NonLinearSolver.MaxSolverIterations = 50;

    // C.LinearSolver = LinearSolverCode.exp_Kcycle_schwarz.GetConfig();


    C.TimesteppingMode = AppControl._TimesteppingMode.Steady;
    C.Timestepper_LevelSetHandling = LevelSetHandling.None;
    C.Option_LevelSetEvolution = LevelSetEvolution.None;

    // C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
    // C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;
    // C.Option_LevelSetEvolution = LevelSetEvolution.StokesExtension;
    // C.TimeSteppingScheme = TimeSteppingScheme.ImplicitEuler;
    // C.dtFixed = 2.5e-5;
    // C.NoOfTimesteps = 4; 

    C.SessionName = $"KovasznayFlow_ConvStudy_VelocityInlet_p{k}_J{J}";
    
    Controls.Add(C);
}
}

In [21]:
Controls.Count

18

In [22]:
//Controls.Select(ctrl => ctrl.SessionName)
Controls.Pick(0).SessionName

KovasznayFlow_ConvStudy_VelocityInlet_p2_J12

In [16]:
int[] numProcs = { 1, 1, 1, 1, 1, 1, 1, 1, 4, 1, 1, 1, 1, 1, 1, 1, 2, 8 };
numProcs.Length

18

In [17]:
myBatch.Name

FDY-WindowsHPC

In [16]:
//Controls.Pick(0).Run();

In [23]:
for (int i = 0; i < Controls.Count; i++) {
    var oneJob              = Controls[i].CreateJob();
    
    oneJob.NumberOfMPIProcs = numProcs[i];

    int numThreads = 1;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

   
    if (myBatch.Name == "Lb2-specialPrj") {
        //oneJob.UseComputeNodesExclusive = true;
        ((SlurmClient)myBatch).ExecutionTime = "24:00:00";
    }

    oneJob.Activate(myBatch); 
}